# 03 — Baseline models

Intended models: Morgan fingerprints with Logistic Regression and Random Forest,
evaluated independently for each CYP isoform. This notebook is a starter scaffold;
run it only after the data and saved split schemas have been confirmed.

In [1]:
!pip install matplotlib


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report
)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
RESULTS_DIR = PROJECT_ROOT / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Cleaned dataset exists:",
      (DATA_DIR / "cyp_substrates_cleaned.csv").exists())
print("Global split exists:",
      (SPLITS_DIR / "cyp_substrates_global_cold_split.csv").exists())


Project root: c:\Users\PAWAR ARAVIND\OneDrive\Desktop\CYP450_Substrate_Liability
Cleaned dataset exists: True
Global split exists: True


In [3]:

data_path = SPLITS_DIR / "cyp_substrates_global_cold_split.csv"
df = pd.read_csv(data_path)

required_columns = {
    "isoform", "label", "smiles_canonical", "global_split"
}
missing_columns = required_columns - set(df.columns)

assert not missing_columns, f"Missing columns: {missing_columns}"
assert df["label"].isin([0, 1]).all()
assert df["smiles_canonical"].notna().all()
assert df["global_split"].isin(["train", "val", "test"]).all()

# Every canonical compound must belong to exactly one partition.
assert (
    df.groupby("smiles_canonical")["global_split"].nunique().max() == 1
)

isoform = "CYP1A2"
data = df[df["isoform"] == isoform].copy()

print("Isoform:", isoform)
print(data.groupby(["global_split", "label"]).size().unstack(fill_value=0))


Isoform: CYP1A2
label         0.0  1.0
global_split          
test           32   27
train         131  122
val            30   24


In [4]:

fingerprint_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

def smiles_to_fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fingerprint = fingerprint_generator.GetFingerprint(mol)
    array = np.zeros((2048,), dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fingerprint, array)
    return array

data["fingerprint"] = data["smiles_canonical"].apply(
    smiles_to_fingerprint
)

invalid_count = data["fingerprint"].isna().sum()
print("Invalid structures:", invalid_count)

data = data.dropna(subset=["fingerprint"]).copy()


Invalid structures: 0


In [5]:

train = data[data["global_split"] == "train"]
val = data[data["global_split"] == "val"]
test = data[data["global_split"] == "test"]

def make_X(partition):
    return np.vstack(partition["fingerprint"].to_numpy())

X_train, y_train = make_X(train), train["label"].astype(int).to_numpy()
X_val, y_val = make_X(val), val["label"].astype(int).to_numpy()
X_test, y_test = make_X(test), test["label"].astype(int).to_numpy()

model = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

val_probabilities = model.predict_proba(X_val)[:, 1]
test_probabilities = model.predict_proba(X_test)[:, 1]

# Choose a classification threshold using validation MCC only.
thresholds = np.arange(0.20, 0.81, 0.01)

threshold_results = [
    (
        threshold,
        matthews_corrcoef(
            y_val, (val_probabilities >= threshold).astype(int)
        )
    )
    for threshold in thresholds
]

best_threshold, best_val_mcc = max(
    threshold_results, key=lambda item: item[1]
)

test_predictions = (test_probabilities >= best_threshold).astype(int)

metrics = {
    "isoform": isoform,
    "model": "Morgan2048_LogisticRegression",
    "split": "global_cold_compound",
    "n_train": len(y_train),
    "n_val": len(y_val),
    "n_test": len(y_test),
    "threshold": float(best_threshold),
    "val_MCC_at_selected_threshold": float(best_val_mcc),
    "test_ROC_AUC": roc_auc_score(y_test, test_probabilities),
    "test_PR_AUC": average_precision_score(y_test, test_probabilities),
    "test_MCC": matthews_corrcoef(y_test, test_predictions),
    "test_F1": f1_score(y_test, test_predictions, zero_division=0),
    "test_accuracy": accuracy_score(y_test, test_predictions),
}

print(pd.Series(metrics).to_string())
print("\nConfusion matrix:")
print(confusion_matrix(y_test, test_predictions))
print("\nClassification report:")
print(classification_report(
    y_test, test_predictions, zero_division=0
))

pd.DataFrame([metrics]).to_csv(
    RESULTS_DIR / "cyp1a2_logistic_regression_metrics.csv",
    index=False
)

pd.DataFrame({
    "smiles_canonical": test["smiles_canonical"].to_numpy(),
    "y_true": y_test,
    "probability_substrate": test_probabilities,
    "prediction": test_predictions
}).to_csv(
    RESULTS_DIR / "cyp1a2_logistic_regression_predictions.csv",
    index=False
)


isoform                                                 CYP1A2
model                            Morgan2048_LogisticRegression
split                                     global_cold_compound
n_train                                                    253
n_val                                                       54
n_test                                                      59
threshold                                                 0.27
val_MCC_at_selected_threshold                         0.445789
test_ROC_AUC                                          0.820602
test_PR_AUC                                           0.783235
test_MCC                                              0.442183
test_F1                                               0.727273
test_accuracy                                         0.694915

Confusion matrix:
[[17 15]
 [ 3 24]]

Classification report:
              precision    recall  f1-score   support

           0       0.85      0.53      0.65        32
          

In [6]:

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=500,
    max_features="sqrt",
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_val_probabilities = rf_model.predict_proba(X_val)[:, 1]
rf_test_probabilities = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest trained successfully.")
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


Random Forest trained successfully.
Train: (253, 2048)
Validation: (54, 2048)
Test: (59, 2048)


In [7]:

rf_threshold_results = [
    (
        threshold,
        matthews_corrcoef(
            y_val,
            (rf_val_probabilities >= threshold).astype(int)
        )
    )
    for threshold in np.arange(0.20, 0.81, 0.01)
]

rf_best_threshold, rf_best_val_mcc = max(
    rf_threshold_results, key=lambda item: item[1]
)

rf_test_predictions = (
    rf_test_probabilities >= rf_best_threshold
).astype(int)

rf_metrics = {
    "isoform": isoform,
    "model": "Morgan2048_RandomForest",
    "split": "global_cold_compound",
    "n_train": len(y_train),
    "n_val": len(y_val),
    "n_test": len(y_test),
    "threshold": float(rf_best_threshold),
    "val_MCC_at_selected_threshold": float(rf_best_val_mcc),
    "test_ROC_AUC": roc_auc_score(y_test, rf_test_probabilities),
    "test_PR_AUC": average_precision_score(
        y_test, rf_test_probabilities
    ),
    "test_MCC": matthews_corrcoef(y_test, rf_test_predictions),
    "test_F1": f1_score(
        y_test, rf_test_predictions, zero_division=0
    ),
    "test_accuracy": accuracy_score(y_test, rf_test_predictions)
}

print(pd.Series(rf_metrics).to_string())
print("\nConfusion matrix:")
print(confusion_matrix(y_test, rf_test_predictions))

pd.DataFrame([rf_metrics]).to_csv(
    RESULTS_DIR / "cyp1a2_random_forest_metrics.csv",
    index=False
)


isoform                                           CYP1A2
model                            Morgan2048_RandomForest
split                               global_cold_compound
n_train                                              253
n_val                                                 54
n_test                                                59
threshold                                           0.49
val_MCC_at_selected_threshold                   0.414371
test_ROC_AUC                                    0.788194
test_PR_AUC                                      0.78178
test_MCC                                         0.35871
test_F1                                         0.666667
test_accuracy                                   0.677966

Confusion matrix:
[[21 11]
 [ 8 19]]


In [8]:

lr_metrics = pd.read_csv(
    RESULTS_DIR / "cyp1a2_logistic_regression_metrics.csv"
)

rf_metrics_df = pd.DataFrame([rf_metrics])

comparison = pd.concat(
    [lr_metrics, rf_metrics_df],
    ignore_index=True
)

comparison_columns = [
    "model",
    "test_ROC_AUC",
    "test_PR_AUC",
    "test_MCC",
    "test_F1",
    "test_accuracy",
    "threshold"
]

print(
    comparison[comparison_columns]
    .sort_values("test_ROC_AUC", ascending=False)
    .to_string(index=False)
)

comparison.to_csv(
    RESULTS_DIR / "cyp1a2_baseline_comparison.csv",
    index=False
)


                        model  test_ROC_AUC  test_PR_AUC  test_MCC  test_F1  test_accuracy  threshold
Morgan2048_LogisticRegression      0.820602     0.783235  0.442183 0.727273       0.694915       0.27
      Morgan2048_RandomForest      0.788194     0.781780  0.358710 0.666667       0.677966       0.49


In [9]:

from sklearn.base import clone

isoforms = ["CYP1A2", "CYP2C19", "CYP2C9", "CYP2D6", "CYP3A4"]

baseline_models = {
    "Morgan2048_LogisticRegression": LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ),
    "Morgan2048_RandomForest": RandomForestClassifier(
        n_estimators=500,
        max_features="sqrt",
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

all_metrics = []
all_predictions = []

for current_isoform in isoforms:
    subset = df[df["isoform"] == current_isoform].copy()

    subset["fingerprint"] = subset["smiles_canonical"].apply(
        smiles_to_fingerprint
    )
    subset = subset.dropna(subset=["fingerprint"])

    train_part = subset[subset["global_split"] == "train"]
    val_part = subset[subset["global_split"] == "val"]
    test_part = subset[subset["global_split"] == "test"]

    def arrays(part):
        X = np.vstack(part["fingerprint"].to_numpy())
        y = part["label"].astype(int).to_numpy()
        return X, y

    Xtr, ytr = arrays(train_part)
    Xv, yv = arrays(val_part)
    Xte, yte = arrays(test_part)

    # ROC-AUC and PR-AUC require both classes in the test partition.
    assert len(np.unique(ytr)) == 2, f"{current_isoform}: train class missing"
    assert len(np.unique(yv)) == 2, f"{current_isoform}: val class missing"
    assert len(np.unique(yte)) == 2, f"{current_isoform}: test class missing"

    for model_name, estimator in baseline_models.items():
        model = clone(estimator)
        model.fit(Xtr, ytr)

        val_probs = model.predict_proba(Xv)[:, 1]
        test_probs = model.predict_proba(Xte)[:, 1]

        threshold_scores = [
            (
                t,
                matthews_corrcoef(yv, (val_probs >= t).astype(int))
            )
            for t in np.arange(0.20, 0.81, 0.01)
        ]

        threshold, val_mcc = max(
            threshold_scores, key=lambda item: item[1]
        )
        test_pred = (test_probs >= threshold).astype(int)

        row = {
            "isoform": current_isoform,
            "model": model_name,
            "split": "global_cold_compound",
            "n_train": len(ytr),
            "n_val": len(yv),
            "n_test": len(yte),
            "threshold": float(threshold),
            "val_MCC": float(val_mcc),
            "test_ROC_AUC": roc_auc_score(yte, test_probs),
            "test_PR_AUC": average_precision_score(yte, test_probs),
            "test_MCC": matthews_corrcoef(yte, test_pred),
            "test_F1": f1_score(yte, test_pred, zero_division=0),
            "test_accuracy": accuracy_score(yte, test_pred)
        }
        all_metrics.append(row)

        for smiles, true_label, probability, prediction in zip(
            test_part["smiles_canonical"].to_numpy(),
            yte, test_probs, test_pred
        ):
            all_predictions.append({
                "isoform": current_isoform,
                "model": model_name,
                "smiles_canonical": smiles,
                "y_true": int(true_label),
                "probability_substrate": float(probability),
                "threshold": float(threshold),
                "prediction": int(prediction)
            })

        print(
            f"{current_isoform} | {model_name} | "
            f"ROC-AUC={row['test_ROC_AUC']:.4f} | "
            f"PR-AUC={row['test_PR_AUC']:.4f} | "
            f"MCC={row['test_MCC']:.4f}"
        )

metrics_df = pd.DataFrame(all_metrics)
predictions_df = pd.DataFrame(all_predictions)

metrics_df.to_csv(
    RESULTS_DIR / "all_isoform_baseline_metrics.csv",
    index=False
)
predictions_df.to_csv(
    RESULTS_DIR / "all_isoform_baseline_predictions.csv",
    index=False
)

print("\nFull baseline comparison:")
print(
    metrics_df.sort_values(
        ["isoform", "test_ROC_AUC"],
        ascending=[True, False]
    ).to_string(index=False)
)


CYP1A2 | Morgan2048_LogisticRegression | ROC-AUC=0.8206 | PR-AUC=0.7832 | MCC=0.4422
CYP1A2 | Morgan2048_RandomForest | ROC-AUC=0.7882 | PR-AUC=0.7818 | MCC=0.3587
CYP2C19 | Morgan2048_LogisticRegression | ROC-AUC=0.7024 | PR-AUC=0.6354 | MCC=0.1979
CYP2C19 | Morgan2048_RandomForest | ROC-AUC=0.7405 | PR-AUC=0.6676 | MCC=0.3381
CYP2C9 | Morgan2048_LogisticRegression | ROC-AUC=0.7774 | PR-AUC=0.6377 | MCC=0.4451
CYP2C9 | Morgan2048_RandomForest | ROC-AUC=0.8082 | PR-AUC=0.7301 | MCC=0.3796
CYP2D6 | Morgan2048_LogisticRegression | ROC-AUC=0.7811 | PR-AUC=0.7656 | MCC=0.4745
CYP2D6 | Morgan2048_RandomForest | ROC-AUC=0.7793 | PR-AUC=0.7423 | MCC=0.3546
CYP3A4 | Morgan2048_LogisticRegression | ROC-AUC=0.7400 | PR-AUC=0.7232 | MCC=0.4516
CYP3A4 | Morgan2048_RandomForest | ROC-AUC=0.7833 | PR-AUC=0.7781 | MCC=0.4235

Full baseline comparison:
isoform                         model                split  n_train  n_val  n_test  threshold  val_MCC  test_ROC_AUC  test_PR_AUC  test_MCC  test_F1  t

## Modeling checklist

- Fit models on training data only; tune thresholds/hyperparameters on validation data.
- Evaluate once on each held-out split.
- Report ROC-AUC as primary, plus PR-AUC, MCC and F1 for every isoform.
- Save test predictions and metric tables.
- Report macro-average ROC-AUC over isoforms with valid test labels.
- Do not report any metrics until the code has actually been run.